In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path

In [ ]:
def open_fits_file(file_path, secondary_header=False):
    """
    Return the primary header and the image data of a FITS file.

    The data is either in extension 0 (most calibration files), in
    extension 1 (raw file), or split with one chip per extension (some
    BLEVFILEs). The returned data always has the shape (chip, y, x).
    If secondary_header is set to True, the 1st & 2nd header are returned: ((header1, header2), data)
    """
    with fits.open(file_path) as hdul:
        header = hdul[0].header.copy()
        header2 = hdul[1].header.copy()
        if hdul[0].data is not None:
            data = hdul[0].data
        elif len(hdul) == 2:
            data = hdul[1].data
        else:
            chips = []
            for i in range(1, len(hdul)):
                chips.append(hdul[i].data)
            data = np.array(chips)
    if secondary_header:
        return (header,header2), data
    else:
        return header, data

In [ ]:
def load_calibration_files(exposure_dir):
    """
    Load every FITS file of one exposure folder into a dict keyed by
    file type, e.g. 'raw', 'maskfile', 'blevfile', ...

    Each value is a (header, data) tuple from open_fits_file().
    For the raw file, 1st & 2nd header are returned: ((header1, header2), data)
    """
    exposure_dir = Path(exposure_dir)
    files = {}
    for file_path in sorted(exposure_dir.glob('*.fits')):
        file_type = file_path.stem.split('_')[-1]
        files[file_type] = open_fits_file(file_path, secondary_header=False)
    return files

In [ ]:
calib_dir = Path("hst-calibration")
ubai2505m = load_calibration_files(calib_dir / "ubai2505m")
raw_header, raw_data = ubai2505m["raw"]

# Logging
print("File type | Data shape |  Data type")
for file_type, (header, data) in ubai2505m.items():
    print(f"{file_type:10s} {str(data.shape):16s} {data.dtype}")
    #print(raw_header)